# 04 · Temporal graph-attention network (Milestone 3)

**Pilot region:** Gulf of Mannar – Palk Bay (`IN-GOM`). **Data:** the same 60-day synthetic simulation and splits as notebook 03.

**The model**
- **Nodes:** each vessel in each 6-hour window.
- **Edges:** (a) proximity encounters, (b) *possible dark encounters*, meaning meetings that could have happened while one vessel was silent, and (c) links to the same vessel 6 h and 12 h earlier (past → present only).
- **Layers:** a TCN + MLP node encoder, then 2 GATv2 layers with edge features and a learned time encoding (PyTorch Geometric).
- **Heads:**
  - fishing activity;
  - suspicious interaction, scored per edge;
  - AIS-gap anomaly;
  - a neural decoder that predicts where a silent vessel went, with a 90% uncertainty region;
  - the final **investigation score**, learned from the node embedding plus all the other heads.

The notebook trains the model and a **no-graph ablation** (same model, all edges removed), retrains the MLP and TCN baselines on the same splits, and compares them. Every loss term is reported separately.

**Runtime:**
- **GPU recommended** (Runtime → Change runtime type → T4 GPU): about 10–15 min in total.
- **CPU only:** about 35–40 min.
- **`FAST_MODE = True`:** 1 seed, about 12 min on CPU.

> ⚠️ **Synthetic demo data.** Scores mean *investigation priority*, never illegality. Graph-attention weights are **not** explanations; proper explanation methods (GNNExplainer/PGExplainer, SHAP, counterfactuals) come in Milestone 4.

In [ ]:
# ---- Setup: works in Google Colab (CPU is enough) and locally --------------------
import os, sys, subprocess, pathlib, zipfile, glob
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Dan-droid-A-113/IUU-Fishing-GNN-India"   # clones the code; set to "" to upload the zip instead
USE_DRIVE = False      # True = keep project + outputs on Google Drive (restart-safe)

if IN_COLAB:
    PROJECT_DIR = pathlib.Path("/content/india-fishing-gnn")
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = pathlib.Path("/content/drive/MyDrive/india-fishing-gnn")
    if not (PROJECT_DIR / "src").exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)
        else:
            # accepts renamed downloads such as "india-fishing-gnn (1).zip"
            zips = sorted(glob.glob("/content/india-fishing-gnn*.zip"), key=os.path.getmtime)
            if not zips:
                from google.colab import files
                print("Project zip not found - choose india-fishing-gnn.zip from your computer:")
                uploaded = files.upload()   # file picker appears below this cell
                zips = [os.path.abspath(n) for n in uploaded if n.endswith(".zip")]
            if not zips:
                raise FileNotFoundError("No project zip uploaded.")
            print("Extracting", zips[-1])
            zipfile.ZipFile(zips[-1]).extractall(PROJECT_DIR.parent)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements-colab.txt")], check=True)
else:
    PROJECT_DIR = pathlib.Path.cwd()
    while not (PROJECT_DIR / "src").exists() and PROJECT_DIR != PROJECT_DIR.parent:
        PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR, "| Colab:", IN_COLAB)

## 0 · Configuration

In [ ]:
import json, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

from src.config import TrainConfig, WindowConfig, FIGURE_DIR, METRIC_DIR, SYNTHETIC_CAVEAT
from src.experiments import (prepare_m2_data, make_splits, run_baselines, run_tgat, merge_results,
                             comparison_table, best_run)
from src.temporal_gnn import GNNConfig, build_graph, graph_summary
from src.graph_builder import EDGE_FEATURES
from src.baselines import get_device
from src import visualization as viz

FAST_MODE = False           # True = 1 seed per model
INCLUDE_BASELINES = True    # retrain MLP + TCN on the same splits for a like-for-like comparison
SEEDS = (0,) if FAST_MODE else (0, 1, 2)
GCFG = GNNConfig(seeds=SEEDS)
DEVICE = get_device()
for p in (FIGURE_DIR, METRIC_DIR):
    p.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| seeds:", SEEDS, "| GPU available:", torch.cuda.is_available())
print(SYNTHETIC_CAVEAT)

## 1 · Data (same 60-day simulation and windows as notebook 03; cached after the first build, ~3 min)

In [ ]:
d = prepare_m2_data()
lab = d["labels"]
display(lab.context.value_counts().rename("windows").to_frame())

## 2 · Build the temporal interaction graph

In [ ]:
G = build_graph(d, WindowConfig(), lags=GCFG.temporal_lags)
display(graph_summary(G).rename("count").to_frame())
print("edge features:", ", ".join(EDGE_FEATURES), "+ 8-d learned time encoding")

## 3 · Splits (identical to notebook 03)

In [ ]:
splits = make_splits(d)
for name, sp in splits.items():
    display(pd.crosstab(pd.Series(sp, name=name), lab.y.rename("synthetic_demo_suspicious")))

## 4 · Train the temporal GAT and the no-graph ablation

- **Training setup:** full-graph training, **inductive**. While fitting, only edges between training nodes are used; validation and test nodes are scored on the full graph without their labels.
- **Early stopping and threshold:** both use validation PR-AUC for the investigation score.
- **Loss:** investigation + fishing + gap-anomaly + interaction-edge BCE, plus 0.3 × the trajectory Gaussian NLL.
- **Speed:** GPU is much faster; CPU works.

In [ ]:
res_g = run_tgat(d, G, splits, GCFG, device=DEVICE)
res_g["runs"].to_csv(METRIC_DIR / "m3_tgat_runs.csv", index=False)
res_g["tasks"].to_csv(METRIC_DIR / "m3_task_metrics.csv", index=False)

## 5 · Baselines on the same splits (MLP and TCN from Milestone 2)

In [ ]:
if INCLUDE_BASELINES:
    res_b = run_baselines(d, splits, TrainConfig(seeds=SEEDS), models=("MLP", "TCN"), device=DEVICE)
    res = merge_results(res_b, res_g)
else:
    res = merge_results(res_g)
table = comparison_table(res["runs"])
table.to_csv(METRIC_DIR / "m3_model_comparison.csv")
display(table)
fig = viz.plot_model_comparison(res["runs"], "pr_auc", FIGURE_DIR / "m3_model_comparison_pr_auc.png"); plt.show()
fig = viz.plot_model_comparison(res["runs"], "f1", FIGURE_DIR / "m3_model_comparison_f1.png", label="Test F1"); plt.show()
fig = viz.plot_pr_curves(res, lab, FIGURE_DIR / "m3_pr_curves.png"); plt.show()

## 6 · Multi-task losses (each term reported separately)

In [ ]:
for sp in splits:
    fig = viz.plot_task_losses(res_g["histories"], sp, path=FIGURE_DIR / f"m3_task_losses_{sp}.png"); plt.show()

## 7 · Auxiliary heads: fishing, gap anomaly, interaction edges, trajectory

- **Fishing, gap-anomaly and interaction-edge heads:** scored by test PR-AUC.
- **Trajectory decoder:** scored on test gaps against the hidden simulated path, which is never used for training.
  - ADE: mean distance over the 8 predicted points.
  - FDE: distance at reappearance.
  - Coverage: share of reappearances inside the predicted 90% region.
- **References:** dead reckoning (last speed and course held) and "stay at the last fix".

In [ ]:
tasks = res_g["tasks"]
cols = ["fishing_pr_auc", "gap_anomaly_pr_auc", "interaction_edge_pr_auc", "traj_ade_km", "traj_fde_km",
        "traj_median_fde_km", "traj_endpoint_90pct_coverage", "traj_n"]
display(tasks.groupby(["split", "model"], sort=False)[[c for c in cols if c in tasks]].mean().round(3))

In [ ]:
# one planted blackout from the test vessels: what the decoder predicts vs the hidden truth
gd = G["gap_ds"]
sp = splits["vessel_disjoint"]
en = gd["end_node"]
cand = np.flatnonzero((en >= 0) & (sp[np.clip(en, 0, None)] == "test") & (G["y"][np.clip(en, 0, None)] == 1)
                      & ~np.isnan(gd["truth_path"]).any(axis=(1, 2)))
best = best_run(res, "vessel_disjoint", "TGAT (temporal graph)")["raw"]
if len(cand):
    k = cand[np.argmax(gd["gaps"].gap_duration_min.values[cand])]
    fig = viz.plot_gap_reconstruction(d, G, best, k, FIGURE_DIR / "m3_gap_reconstruction_example.png",
                                      title="Planted synthetic blackout (test vessel) - neural gap reconstruction")
    plt.show()

## 8 · Does the graph help? Hard negatives and event types

In [ ]:
from src.evaluation import binary_metrics
rows = []
for sp_name in splits:
    t = viz.fp_by_context(res, lab, sp_name)
    t.to_csv(METRIC_DIR / f"m3_false_alerts_by_context_{sp_name}.csv", index=False)
    fig = viz.plot_fp_by_context(t, sp_name, FIGURE_DIR / f"m3_false_alerts_by_context_{sp_name}.png"); plt.show()
    display(t.pivot(index="context", columns="model", values="fp_rate").round(3))

ev_rows = []
for (sp_name, m, seed), p in res["preds"].items():
    for et, g in p["events"].groupby("event_type"):
        ev_rows.append({"split": sp_name, "model": m, "event_type": et, "recall": g.detected.mean()})
ev_tab = pd.DataFrame(ev_rows).groupby(["split", "event_type", "model"]).recall.mean().unstack("model").round(2)
ev_tab.to_csv(METRIC_DIR / "m3_event_recall_by_type.csv")
display(ev_tab)

In [ ]:
fig = viz.plot_fp_gallery(d, best_run(res, "vessel_disjoint", "TGAT (temporal graph)"), "vessel_disjoint",
                          "TGAT (temporal graph)", FIGURE_DIR / "m3_false_alert_gallery_TGAT.png")
if fig is not None:
    plt.show()

## 9 · Save summary

In [ ]:
summary = {
    "milestone": 3, "data_mode": "A_synthetic (60 days)", "caveat": SYNTHETIC_CAVEAT, "device": str(DEVICE),
    "graph": graph_summary(G).to_dict(), "gnn_config": vars(GCFG),
    "comparison_mean": res["runs"].groupby(["split", "model"])[["pr_auc", "f1", "precision", "recall",
        "false_alerts_per_1000_vessel_h", "event_recall"]].mean().round(4).reset_index().to_dict("records"),
    "task_metrics_mean": tasks.groupby(["split", "model"]).mean(numeric_only=True).round(4).reset_index().to_dict("records"),
}
(METRIC_DIR / "m3_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("metrics:", *sorted(p.name for p in METRIC_DIR.glob("m3_*")), sep="\n  ")
print("figures:", *sorted(p.name for p in FIGURE_DIR.glob("m3_*")), sep="\n  ")

## How to read the results, and what Milestone 3 can and cannot conclude

**Questions to answer from the tables above**

1. **Does the graph add information?** Compare *TGAT (temporal graph)* with *TGAT without graph (ablation)*. They are the same model trained the same way, and only the edges differ. A higher PR-AUC and fewer false alerts, especially on *carrier loitering* and *benign_gap* windows and on *low_speed_rendezvous* event recall, would mean neighbour messages help. If the two are tied, the graph adds nothing yet on this data, and that should be reported honestly.
2. **Is it better than the strongest non-graph baseline (MLP)?** The MLP is strong on this synthetic data because its hand-built gap and encounter aggregates closely match how suspicious behaviour was simulated.
3. **Does the trajectory decoder beat dead reckoning?** Compare ADE and FDE. A 90% coverage close to 0.90 means the uncertainty regions are well calibrated.

**Can conclude:** whether, on this synthetic pilot, the temporal interaction graph and the multi-task heads improve the investigation score over non-graph models, with all runs using leakage-safe splits.

**Cannot conclude:**
- anything about real vessels or real illegal fishing;
- anything from raw attention weights, which are not explanations;
- performance on real AIS, or transfer to other Indian regions.

**Next, Milestone 4:** the explanation layer. It covers GNNExplainer/PGExplainer for important nodes and edges, SHAP for features, temporal masking, counterfactual tests, and fidelity/sparsity/stability checks. It also produces structured JSON alerts with a caveat.